In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

# Main libraries
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# We will use Polars for data manipulation
import polars as pl

from config import DATA_PATH
from data import SPECIES_MAPPING

In [ ]:
df = pl.read_parquet(
    # os.path.join(DATA_PATH, "tidy", "cpf-level2_growth-periods_with-cc.parquet")
    os.path.join(DATA_PATH, "tidy", "cpf-level2_cleaned.parquet")
).with_columns(species=pl.col("specie").cast(pl.Utf8).replace(SPECIES_MAPPING))

var_df = df.select(
    ["plot_id", "plot_latitude", "plot_longitude", "growth_rate_rel", "species"]
).to_pandas()

In [ ]:
from scipy.spatial.distance import pdist, squareform


# Aggregate to one value per plot (mean growth rate)
plot_means = (
    var_df.groupby(["plot_id", "plot_latitude", "plot_longitude"])
    .agg(growth_rate_mean=("growth_rate_rel", "mean"))
    .reset_index()
)

# Calculate pairwise distances (Euclidean in meters)
coords = plot_means[["plot_latitude", "plot_longitude"]].values
dist_matrix = squareform(pdist(coords, metric="euclidean"))

# Calculate variogram: 0.5 * E[(Z(x) - Z(x+h))^2]
values = plot_means["growth_rate_mean"].values
n = len(values)

# Create distance bins
max_dist = dist_matrix.max()
bins = np.linspace(0, max_dist, 25)
bin_centers = (bins[:-1] + bins[1:]) / 2
variogram = []

for i in range(len(bins) - 1):
    mask = (dist_matrix > bins[i]) & (dist_matrix <= bins[i + 1])
    if mask.sum() > 0:
        diff = values[:, None] - values
        gamma = 0.5 * (diff[mask] ** 2).mean()
        variogram.append(gamma)
    else:
        variogram.append(np.nan)

# Plot variogram
plt.figure(figsize=(8, 5))
plt.plot(
    bin_centers[~np.isnan(variogram)],
    np.array(variogram)[~np.isnan(variogram)],
    "o-",
    color="darkgreen",
)
plt.xlabel("Distance between plots (meters)")
plt.ylabel("Semivariance")
plt.title("Empirical Variogram of Growth Rate")
plt.grid(True)
plt.show()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Aggregate mean growth rate per plot
plot_data = (
    var_df.groupby(["plot_id", "plot_latitude", "plot_longitude"])
    .agg(mean_growth=("growth_rate_rel", "mean"))
    .reset_index()
)

# Convert to GeoDataFrame
geometry = [
    Point(xy) for xy in zip(plot_data["plot_longitude"], plot_data["plot_latitude"])
]
gdf = gpd.GeoDataFrame(
    plot_data, geometry=geometry, crs="EPSG:4326"
)  # adjust CRS if needed

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    gdf["plot_longitude"],
    gdf["plot_latitude"],
    c=gdf["mean_growth"],
    cmap="viridis",
    s=50,
    edgecolor="black",
)
plt.colorbar(sc, label="Mean growth rate (relative)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Spatial distribution of mean growth rate")
plt.show()

In [ ]:
import skgstat as skg

# Aggregate to one value per plot (mean growth rate)
plot_means = (
    var_df.groupby(["plot_id", "plot_latitude", "plot_longitude", "species"])
    .agg(growth_rate_mean=("growth_rate_rel", "mean"))
    .reset_index()
)

plot_means = plot_means[plot_means["species"].isin(list(SPECIES_MAPPING.values()))]

for specie in plot_means["species"].unique():
    tdf = plot_means[plot_means["species"] == specie]
    coords = tdf[["plot_latitude", "plot_longitude"]].values
    values = tdf["growth_rate_mean"].values
    V = skg.Variogram(coords, values, n_lags=25, normalize=False)
    fig = V.plot()
    fig.suptitle(specie.capitalize())
    fig.show()

    fig = V.distance_difference_plot()
    fig.suptitle(specie.capitalize())
    fig.show()